In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
import xgboost as xgb

# ==============================
# 预加载静态数据
# ==============================
risk_factor_df = pd.read_excel(r'output/1_risk_factor/risk_factor.xlsx')
weight_df = pd.read_excel("output/2_weight/weight.xlsx")[["gb", "rescaled"]]

# ===========================
#       文件保存辅助函数
# ===========================
def save_ci(values, target_species, outname):
    df = pd.DataFrame({
        "index": ["mean", "lbd", "ubd"],
        "value": [np.mean(values), np.percentile(values, 2.5), np.percentile(values, 97.5)]
    })
    outdir = Path(f"output/4_predict/pathogen/{outname}")
    outdir.mkdir(parents=True, exist_ok=True)
    filename = f"{target_species}.xlsx"
    df.to_excel(outdir / filename, index=False)

def save_df(df, target_species,outdir_name):
    outdir = Path(f"output/4_predict/pathogen/{outdir_name}")
    outdir.mkdir(parents=True, exist_ok=True)
    filename = f"{target_species}.xlsx"
    df.to_excel(outdir / filename, index=False)

# ===========================
#         BRT 主函数
# ===========================
def ran_BRT(target_species,
            learning_rate,
            max_depth,
            n_estimators,
            subsample,
            risk_factor_df,
            weight_df,
            case_control_df):

    all_risk_cols = risk_factor_df.columns.drop("gb")
    all_data = (
        risk_factor_df
        .merge(weight_df, on="gb", how="left")
        .merge(case_control_df[["gb", target_species]], on="gb", how="left")
        .rename(columns={target_species: "status"})
    )
    survey_data = all_data.dropna(subset=["status"]).reset_index(drop=True)
    all_x = all_data[all_risk_cols].values
    all_gb = all_data["gb"].values

    importance_dict = {k: [] for k in all_risk_cols}
    auc_list, tpr_list, tnr_list = [], [], []
    preds_list = []

    xgb_params = dict(
        objective="binary:logistic",
        eval_metric="logloss",
        learning_rate=float(learning_rate),
        max_depth=int(max_depth),
        subsample=float(subsample),
        n_estimators=int(n_estimators)
    )

    for i in range(100):
        train, test = train_test_split(
            survey_data,
            test_size=0.25,
            random_state=i,
            stratify=survey_data["status"]
        )
        train_x = train[all_risk_cols].values
        train_y = train["status"].values
        train_w = train["rescaled"].values

        scale_pos_weight = sum(train_y == 0) / sum(train_y == 1)
        model = xgb.XGBClassifier(**xgb_params, random_state=i, scale_pos_weight=scale_pos_weight)
        model.fit(train_x, train_y, sample_weight=train_w)

        # Feature importance
        for idx, k in enumerate(all_risk_cols):
            importance_dict[k].append(model.feature_importances_[idx])

        # ROC
        test_pred = model.predict_proba(test[all_risk_cols].values)[:, 1]
        fpr, tpr, thresholds = roc_curve(test["status"], test_pred)

        # 排除 inf 或 nan
        valid_idx = np.isfinite(thresholds)
        fpr = fpr[valid_idx]
        tpr = tpr[valid_idx]
        thresholds = thresholds[valid_idx]

        # Youden index 或 F1 最大化
        youden_idx = np.argmax(tpr - fpr)
        best_threshold = thresholds[youden_idx]

        auc_list.append(auc(fpr, tpr))
        tpr_list.append(tpr[youden_idx])
        tnr_list.append(1 - fpr[youden_idx])

        all_pred = model.predict_proba(all_x)[:, 1]
        preds_list.append(pd.DataFrame({
            "round": i,
            "threshold": best_threshold,
            "gb": all_gb,
            "pred": all_pred
        }))

    # 保存结果
    save_ci(auc_list, target_species, "auc")
    save_ci(tpr_list, target_species, "tpr")
    save_ci(tnr_list, target_species, "tnr")

    rel_con = pd.DataFrame(importance_dict).T.reset_index().rename(columns={"index": "Key"})
    rel_con["row_sum"] = rel_con.iloc[:, 1:].sum(axis=1)
    rel_con["std_sum"] = rel_con["row_sum"] / rel_con["row_sum"].sum()
    save_df(rel_con, target_species, "rc")

    pred_df = pd.concat(preds_list, ignore_index=True)
    save_df(pred_df, target_species, "prob")

    print(f"✓ BRT finished → {target_species}")

# ===========================
#       批量运行
# ===========================
grid = pd.read_excel("output/3_grid_search/pathogen_best_params.xlsx")

for _, row in grid.iterrows():
    target_species = row["species"]
    learning_rate = row["learning_rate"]
    max_depth = row["max_depth"]
    n_estimators = row["n_estimators"]
    subsample = row["subsample"]
    case_control_df = pd.read_excel(f"output/4_predict/pathogen/case_control/case_control_{target_species}.xlsx")
    ran_BRT(target_species, learning_rate, max_depth, n_estimators, subsample, risk_factor_df=risk_factor_df, weight_df=weight_df, case_control_df=case_control_df)
    # print(f"✓ {target_species} all scenarios done.")

✓ BRT finished → Hantaan virus
✓ BRT finished → Seoul virus
✓ BRT finished → Wenzhou arenavirus
✓ BRT finished → Encephalomyocarditis virus
✓ BRT finished → Rat hepatitis E virus
✓ BRT finished → Betacoronavirus
✓ BRT finished → Morbillivirus
✓ BRT finished → Henipavirus
